In [4]:
import os
import glob
import numpy as np
import h5py
from scipy.io import wavfile
from scipy.signal import hilbert
from mne.filter import filter_data, resample
import mne
from config import DATA_ROOT, ALIGNMENT, AUDIOBOOKS, print_versions
print_versions()
mne.set_log_level("WARNING")      # hides MNE's long filter messages

ROOT = str(DATA_ROOT)
OUT  = os.path.join(ROOT, "Preprocessed", "P05")

FS_EEG_RAW = 1000
FS_AUDIO   = 16000
FS_TARGET  = 125
EEG_LOW    = 0.5
EEG_HIGH   = 8.0
N_ELECTRODES = 64          # 63 recorded + the PO4 reference

os.makedirs(OUT, exist_ok=True)

with h5py.File(os.path.join(ROOT, "P05.h5"), 'r') as f:
    ch_names = [c.decode() for c in f['ch_names'][:]]

with open(os.path.join(OUT, "ch_names.txt"), 'w') as f:
    f.write("\n".join(ch_names))

wavs = sorted(glob.glob(os.path.join(ROOT, "audiobooks", "*.wav")))
assert len(wavs) == 15

for i, wav_path in enumerate(wavs):

    chapter = os.path.splitext(os.path.basename(wav_path))[0]

    # --- envelope, as in the authors' preprocess.py -------------------
    fs, audio = wavfile.read(wav_path)
    assert fs == FS_AUDIO

    envelope = np.abs(hilbert(audio))
    envelope = resample(envelope, 1, FS_AUDIO // FS_TARGET)
    envelope = filter_data(envelope, sfreq=FS_TARGET, l_freq=None, h_freq=50)

       # --- EEG ----------------------------------------------------------
    with h5py.File(os.path.join(ROOT, "P05.h5"), 'r') as f:
        eeg = f[f'data/{chapter}'][:]                 # (63, n) volts, 1000 Hz

    # 1. Average reference over all 64 electrodes (at 1000 Hz)
    eeg = eeg - eeg.sum(axis=0, keepdims=True) / N_ELECTRODES

    # 2. FILTER FIRST (at the raw sampling rate of 1000 Hz) to prevent aliasing
    eeg = filter_data(eeg, sfreq=FS_EEG_RAW, l_freq=EEG_LOW, h_freq=EEG_HIGH)

    # 3. DOWNSAMPLE SECOND (from 1000 Hz to 125 Hz)
    eeg = resample(eeg, 1, FS_EEG_RAW // FS_TARGET)
    
    # 4. Align timelines
    eeg = eeg[:, :len(envelope)]

    # 5. Scale from Volts to Microvolts (Keep z-score commented out for ERPs!)
    eeg = eeg * 1e6 

    np.save(os.path.join(OUT, f"eeg_part{i:02d}.npy"),  eeg)
    np.save(os.path.join(OUT, f"stim_part{i:02d}.npy"), envelope)

    # Updated "SD" to "µV" in the print statement since z-score is off
    print(f"part{i:02d}  {chapter}  eeg {eeg.shape}  stim {envelope.shape}  "
          f"peak {np.abs(eeg).max():.1f} µV")


print("\nwritten to:", OUT)

Python 3.12.13 | numpy 2.5.3 | scipy 1.18.1 | sklearn 1.9.0 | mne 1.12.1 | h5py 3.16.0 | matplotlib 3.11.1
part00  AUNP01  eeg (63, 20271)  stim (20271,)  peak 252.4 µV
part01  AUNP02  eeg (63, 19886)  stim (19886,)  peak 237.5 µV
part02  AUNP03  eeg (63, 18320)  stim (18320,)  peak 218.7 µV
part03  AUNP04  eeg (63, 17926)  stim (17926,)  peak 221.4 µV
part04  AUNP05  eeg (63, 24218)  stim (24218,)  peak 239.5 µV
part05  AUNP06  eeg (63, 18390)  stim (18390,)  peak 225.5 µV
part06  AUNP07  eeg (63, 23081)  stim (23081,)  peak 267.8 µV
part07  AUNP08  eeg (63, 22500)  stim (22500,)  peak 232.1 µV
part08  BROP01  eeg (63, 20795)  stim (20795,)  peak 235.1 µV
part09  BROP02  eeg (63, 22814)  stim (22814,)  peak 214.1 µV
part10  BROP03  eeg (63, 20474)  stim (20474,)  peak 267.7 µV
part11  FLOP01  eeg (63, 16143)  stim (16143,)  peak 246.1 µV
part12  FLOP02  eeg (63, 14598)  stim (14598,)  peak 221.2 µV
part13  FLOP03  eeg (63, 13724)  stim (13724,)  peak 226.2 µV
part14  FLOP04  eeg (63, 

In [5]:
import os
import re
import json

ALIGN = str(ALIGNMENT)
OUT_DIR = str(AUDIOBOOKS)

os.makedirs(OUT_DIR, exist_ok=True)


def parse_textgrid(path):
    """Parse a long-format Praat TextGrid into {tier_name: [intervals]}.

    Interval fields are only read while inside an `intervals [n]:` block, so
    the file-level and tier-level xmin/xmax bounds cannot be mistaken for
    interval times. Each interval is appended on its `text =` line, while it
    is still in scope, so a tier boundary cannot capture a pending interval.
    """
    with open(path, encoding="utf-8") as f:
        lines = [l.strip() for l in f]

    tiers = {}
    declared = {}
    tier = None
    in_interval = False
    xmin = xmax = None

    for line in lines:
        if line.startswith('name ='):
            tier = line.split('=', 1)[1].strip().strip('"')
            tiers[tier] = []
            in_interval = False

        elif line.startswith('intervals: size ='):
            declared[tier] = int(line.split('=')[1])

        elif re.match(r'intervals \[\d+\]:', line):
            in_interval = True
            xmin = xmax = None

        elif in_interval and line.startswith('xmin ='):
            xmin = float(line.split('=')[1])

        elif in_interval and line.startswith('xmax ='):
            xmax = float(line.split('=')[1])

        elif in_interval and line.startswith('text ='):
            label = line.split('=', 1)[1].strip().strip('"')
            tiers[tier].append({"start": xmin, "end": xmax, "label": label})
            in_interval = False

    for name in tiers:
        assert len(tiers[name]) == declared[name], \
            f"{os.path.basename(path)} [{name}]: got {len(tiers[name])}, " \
            f"file declares {declared[name]}"

    return tiers


for chapter in sorted(os.listdir(ALIGN)):

    chapter_dir = os.path.join(ALIGN, chapter)
    if not os.path.isdir(chapter_dir):
        continue

    path = os.path.join(chapter_dir, f"{chapter}.TextGrid")
    if not os.path.exists(path):
        print(f"Warning: missing TextGrid for {chapter}")
        continue

    tiers = parse_textgrid(path)

    with open(os.path.join(OUT_DIR, f"{chapter}_annotations.json"), "w", encoding="utf-8") as f:
        json.dump(tiers, f, indent=2)

    print(f"{chapter:<10} {len(tiers['phones']):>5} phones  "
          f"{len(tiers['words']):>4} words  "
          f"ends {tiers['phones'][-1]['end']:.2f} s")

print("\nWritten to:", OUT_DIR)

AUNP01      1613 phones   462 words  ends 162.15 s
AUNP02      1574 phones   465 words  ends 159.06 s
AUNP03      1334 phones   421 words  ends 146.54 s
AUNP04      1407 phones   438 words  ends 143.39 s
AUNP05      1899 phones   581 words  ends 193.73 s
AUNP06      1436 phones   433 words  ends 147.10 s
AUNP07      1665 phones   513 words  ends 184.63 s
AUNP08      1674 phones   531 words  ends 179.98 s
BROP01      1543 phones   471 words  ends 166.34 s
BROP02      1826 phones   589 words  ends 182.50 s
BROP03      1655 phones   513 words  ends 163.78 s
FLOP01      1348 phones   430 words  ends 129.13 s
FLOP02      1179 phones   377 words  ends 116.77 s
FLOP03      1096 phones   342 words  ends 109.77 s
FLOP04      1918 phones   610 words  ends 196.95 s

Written to: C:\Users\frank\OneDrive\Desktop\Correlating Brain responses with Audio Features\FinalNotebook\Data\Cortical Tracking of Surprisal during Continuous Speech Comprehension\audiobooks


In [6]:
import os
import numpy as np
from scipy.io import wavfile
import json

ROOT = str(DATA_ROOT)
ALIGN = os.path.join(ROOT, "stories", "story_parts", "alignement_data")
AUDIO = os.path.join(ROOT, "audiobooks")
P05   = os.path.join(ROOT, "Preprocessed", "P05")
ANNOT = os.path.join(ROOT, "audiobooks")

chapters = sorted(os.listdir(ALIGN))

print(f"{'chapter':8} {'align_wav':>12} {'book_wav':>10} {'eeg':>10} {'tg_xmax':>9}")

for i, ch in enumerate(chapters):
    fs_a, a = wavfile.read(os.path.join(ALIGN, ch, f"{ch}.wav"))
    fs_b, b = wavfile.read(os.path.join(AUDIO, f"{ch}.wav"))

    eeg = np.load(os.path.join(P05, f"eeg_part{i:02d}.npy"), mmap_mode='r')

    with open(os.path.join(ANNOT, f"{ch}_annotations.json")) as f:
        tg = json.load(f)

    print(f"{ch:8} {len(a)/fs_a:9.2f}s@{fs_a//1000}k "
          f"{len(b)/fs_b:9.2f}s "
          f"{eeg.shape[1]/125:9.2f}s "
          f"{tg['phones'][-1]['end']:8.2f}s")

chapter     align_wav   book_wav        eeg   tg_xmax
AUNP01      162.17s@16k    162.17s    162.17s   162.15s
AUNP02      159.08s@16k    159.08s    159.09s   159.06s
AUNP03      146.56s@16k    146.56s    146.56s   146.54s
AUNP04      143.41s@16k    143.41s    143.41s   143.39s
AUNP05      193.75s@16k    193.75s    193.74s   193.73s
AUNP06      147.12s@16k    147.12s    147.12s   147.10s
AUNP07      184.65s@16k    184.65s    184.65s   184.63s
AUNP08      180.00s@16k    180.00s    180.00s   179.98s
BROP01      166.36s@16k    166.36s    166.36s   166.34s
BROP02      182.51s@16k    182.51s    182.51s   182.50s
BROP03      163.79s@16k    163.79s    163.79s   163.78s
FLOP01      129.15s@16k    129.15s    129.14s   129.13s
FLOP02      116.78s@16k    116.78s    116.78s   116.77s
FLOP03      109.79s@16k    109.79s    109.79s   109.77s
FLOP04      196.97s@16k    196.97s    196.98s   196.95s


In [7]:
# --- Amplitude screening of the preprocessed EEG (µV, 0.5-8 Hz, 125 Hz) ---
import os
import numpy as np

ROOT = str(DATA_ROOT)
P05  = os.path.join(ROOT, "Preprocessed", "P05")

ch_names = open(os.path.join(P05, "ch_names.txt")).read().split()
eeg = np.concatenate([np.load(os.path.join(P05, f"eeg_part{i:02d}.npy"))
                      for i in range(15)], axis=1)          # (63, samples)

sd    = eeg.std(axis=1)                                     # per-channel SD
peak  = np.abs(eeg).max(axis=1)                             # per-channel peak
frac  = (np.abs(eeg) > 100).mean(axis=1) * 100              # % samples > 100 µV
ratio = sd / np.median(sd)

print(f"median channel SD {np.median(sd):.1f} µV   "
      f"range {sd.min():.1f} to {sd.max():.1f} µV\n")

print("Channels ranked by SD")
print(f"{'ch':6} {'SD µV':>7} {'x median':>9} {'peak µV':>8} {'% >100µV':>9}")
for c in np.argsort(sd)[::-1][:8]:
    print(f"{ch_names[c]:6} {sd[c]:7.1f} {ratio[c]:9.2f} "
          f"{peak[c]:8.1f} {frac[c]:9.3f}")

flag = [ch_names[c] for c in np.where(ratio > 3)[0]]
print("\nChannels with SD > 3 x median:", flag if flag else "none")
print("Channel holding the overall peak:", ch_names[int(peak.argmax())])
print(f"FC5   SD {sd[ch_names.index('FC5')]:.1f} µV   "
      f"x median {ratio[ch_names.index('FC5')]:.2f}")

median channel SD 7.7 µV   range 2.8 to 40.3 µV

Channels ranked by SD
ch       SD µV  x median  peak µV  % >100µV
Fp1       40.3      5.23    267.8     5.321
Fp2       36.9      4.78    224.6     4.765
AF4       28.7      3.73    185.8     2.857
AF7       26.0      3.37    213.4     1.694
TP10      20.6      2.68    175.7     0.318
AF8       19.2      2.49    117.1     0.041
F8        17.9      2.32    112.6     0.009
AFz       16.9      2.19    108.8     0.003

Channels with SD > 3 x median: ['AF4', 'AF7', 'Fp1', 'Fp2']
Channel holding the overall peak: Fp1
FC5   SD 3.4 µV   x median 0.44
